# 01 — TiNA Data Understanding

**Goal:** verify the source schema, time coverage, missingness, and maintenance/alarm distributions before preprocessing.

The official source is [ari-dasci/OD-TINA](https://github.com/ari-dasci/OD-TINA). The dataset is too large to load eagerly on a typical laptop, so this notebook starts with a bounded preview. Do not commit source data or notebook outputs containing large tables.

## Reproducibility record

Verified Week 2 pilot record:

- Source: `ari-dasci/OD-TINA` commit `927b1584ea4aeac2ac7daa084a82f899b114419b`; downloaded 2026-09-21.
- Local file: `data/raw/tina_sample_2017-09-08_to_10.parquet`; SHA-256 `19365a40e99083c37c9909f25b281caad6e47430c52b255c4c24b0db874db947`.
- Selected period: 2017-09-08 00:00:00 through 2017-09-10 23:59:59 UTC. This contiguous three-day window fits in laptop memory and contains maintenance and alarm activity.
- Analyst: PROJ 613 project team (Ganesh Mishra, Lakshya Sharma, and Sagar Patel).

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_utils import resolve_data_path, scan_tina, validate_event_columns

DATA_PATH = resolve_data_path()
PREVIEW_ROWS = int(os.environ.get('TINA_PREVIEW_ROWS', '300000'))
print(f'Data path: {DATA_PATH}')
print(f'Bounded preview rows: {PREVIEW_ROWS:,}')

## Load lazily and validate

The preview supports quick iteration. Final claims must be rerun against a documented, event-preserving time period rather than an arbitrary row sample.

In [ ]:
lazy = scan_tina(DATA_PATH)
validate_event_columns(lazy)
schema = lazy.collect_schema()
print(f'Columns: {len(schema.names())}')
schema

In [ ]:
preview = lazy.head(PREVIEW_ROWS).collect()
print(f'Preview shape: {preview.shape}')
preview.head()

## Time field and ordering

Confirm the real timestamp column manually if it is not detected. Never build lags, rolling features, or train/test splits until ascending time order is verified.

In [ ]:
TIME_CANDIDATES = ('unix', 'timestamp', 'datetime', 'date', 'time', 'Date', 'Time')
time_col = next((name for name in TIME_CANDIDATES if name in preview.columns), None)
print(f'Detected time column: {time_col!r}')
if time_col:
    print(preview.select(pl.col(time_col).min().alias('min_time'), pl.col(time_col).max().alias('max_time')))
    print(f'Ascending in preview: {preview[time_col].is_sorted()}')
else:
    print('Set time_col to the verified timestamp field before continuing to time-based analysis.')

## Missingness and constant fields

In [ ]:
quality = pl.DataFrame({
    'column': preview.columns,
    'dtype': [str(dtype) for dtype in preview.dtypes],
    'null_count': [preview[column].null_count() for column in preview.columns],
    'n_unique': [preview[column].n_unique() for column in preview.columns],
}).with_columns((pl.col('null_count') / preview.height * 100).alias('null_pct'))
quality.sort('null_pct', descending=True).head(25)

## Maintenance and alarm distributions

In [ ]:
for column in ('m_id', 'm_subid', 'alarms'):
    display(
        preview.group_by(column)
        .len(name='rows')
        .with_columns((pl.col('rows') / preview.height * 100).alias('pct'))
        .sort('rows', descending=True)
        .head(20)
    )

## Initial decisions

Verified findings and provisional decisions:

- Verified timestamp and sampling interval: `unix` in UTC seconds; 259,200 unique, ascending timestamps at an exact one-second cadence.
- Data-quality issues requiring treatment: no null, NaN, or infinite values in the pilot; eight constant sensor columns require reassessment on broader data; categorical sensors and event codes require explicit encoding; class imbalance is material.
- Maintenance/alarm codes present: `m_id` M4/M9; `m_subid` MS3/MS5/MS6; alarms A5/A12/A13/A23/A34/A35/A95/A96/A97/A111/A115/A116/A118/A128.
- Event-preserving subset and memory estimate: 2017-09-08 through 2017-09-10, 259,200 rows, 4,994 maintenance rows, 26,719 alarm rows, about 2.5 MB Parquet and 220.7 MB in memory.
- Candidate prediction horizons: provisional 30 minutes, 1 hour, and 4 hours; finalize during Week 6 target design.
- Recovery/exclusion rule: exclude rows with active maintenance; test a provisional 30-minute post-maintenance recovery window during Week 3 event-duration analysis.